# CV Masterclass 04: Segmentation & Localization

Welcome to Notebook 04. Object detection draws a box. Segmentation is pixel-perfect intelligence. It classifies every single pixel in an image into a distinct semantic class.

We will explore the legendary U-Net architecture, the mechanics of Decoder upsampling, Atrous Pyramids, and how Mask R-CNN achieved instance-level boundary predictions using Sub-Pixel Alignment.

---

## 🎓 Socratic Deep Check
Ponder this question before we begin. The answer involves the physics of spatial compression:

> *"In a U-Net, the Encoder compresses the image into a tiny $32\times32$ feature map with 1024 channels. The Decoder expands this back up to the original full image resolution. What exact mathematical spatial information is permanently destroyed during the compression phase that forces the architecture to use 'Skip Connections' to restore the high-res boundaries?"*

---

## Table of Contents
1. Semantic vs Instance Segmentation
2. FCN: Historical Foundation
3. The Autoencoder Flaw
4. The U-Net Innovation (+ Attention U-Net)
5. Upsampling Mechanics
6. ASPP (Atrous SPP)
7. Evaluation: mIoU
8. Dice and Tversky Loss
9. Medical Segmentation: Boundary Loss
10. Mask R-CNN
11. Panoptic Segmentation & PQ
12. Mask2Former: Universal Mask Classification
13. Deep Socratic Synthesis


## 1. Semantic vs. Instance Segmentation

*   **Semantic Segmentation:** Classifies pixels by category. If there are 5 sheep in a field, Semantic Segmentation colors all pixels belonging to *any* sheep blue. It does not know where one sheep ends and another begins.
*   **Instance Segmentation (e.g., Mask R-CNN):** Identifies individual objects. It colors Sheep A red, Sheep B green, and Sheep C blue. It is essentially Object Detection + Semantic Segmentation.

### ⚠️ Common Pitfalls
*   **Assuming semantic overlaps:** If an autonomous driving model utilizes only Semantic Segmentation, and two pedestrians walk closely while slightly overlapping, the vision matrix perceives them as one single giant macroscopic blob of human class, completely failing to identify the entity separation.

## 2. FCN: The Historical Foundation

In 2014, classification networks like VGG-16 traditionally ended with three "Fully Connected" (FC) layers, preceded by a Flatten operation that instantly permanently destroyed all 2D spatial dimensions.

**FCN's profound insight:** Fully connected layers are mathematically identical to $1\times1$ convolutions computed over the entire unified spatial map.
If VGG ends with `FC(4096, 1000)` outputting a 1000-class vector, you can delete the Flatten layer and replace it with `Conv2d(in_channels=4096, out_channels=1000, kernel_size=1)`.

By preserving the 2D grid, a `Conv2d(K, 1, 1)` dynamically applied to an extracted deep feature block of shape $(B, K, H, W)$ evaluates mathematically into $(B, 1, H, W)$—a raw class heatmap!

In [ ]:
import torch
import torch.nn as nn

# Simulating FCN-32s
class SimpleFCN32s(nn.Module):
    def __init__(self, num_classes=21):
        super(SimpleFCN32s, self).__init__()
        
        # 1. Simulating VGG Extractor. Spatial dims shrink by 32x due to 5 MaxPool layers.
        # e.g., 224x224 image -> 7x7 deep block.
        self.vgg_backbone = nn.Sequential(
            nn.Conv2d(3, 512, kernel_size=3, padding=1),
            nn.MaxPool2d(kernel_size=32, stride=32)
        )
        
        # 2. Re-defining the FC layers as 1x1 Spatial Convolutions
        self.classifier_conv = nn.Conv2d(512, num_classes, kernel_size=1)
        
        # 3. Massively Upsample the tiny 7x7 heatmap back to 224x224 (stride=32)
        self.upsample = nn.ConvTranspose2d(num_classes, num_classes, kernel_size=64, stride=32, padding=16)

    def forward(self, x):
        deep_features = self.vgg_backbone(x)
        # Spatial heatmap evaluation
        class_heatmaps = self.classifier_conv(deep_features)
        # Restore reality
        return self.upsample(class_heatmaps)

# -----------------------------------------------------
# TEST IT
# -----------------------------------------------------
dummy_image = torch.randn(1, 3, 224, 224)
fcn_model = SimpleFCN32s(num_classes=21)

pixel_predictions = fcn_model(dummy_image)

print(f"FCN-32s Original   Image: {dummy_image.shape}")
print(f"FCN-32s Prediction Image: {pixel_predictions.shape}")
assert pixel_predictions.shape == (1, 21, 224, 224), "FCN-32s failed to restore full image mask!"


### The Blurriness Problem
Because FCN-32s linearly drags a $7\times7$ grid back into a $224\times224$ canvas, the resultant mask is incredibly soft and heavily pixelated. Predicting structural edges smoothly from massively shrunken feature maps is physically impossible. This constraint forces the development of U-Net!

### ⚠️ Common Pitfalls
*   **FCN Stride Alignment:** In `ConvTranspose2d`, computing exact output configurations given large strides requires heavily tuned `padding` matrices. If output padding isn't calculated carefully, expanding a $7\times7$ tensor with $stride=32$ might output $228\times228$ rather than aligning identically to the original $224\times224$ constraint.

## 3. The Autoencoder Flaw

Early segmentation networks attempted an Autoencoder logic: Compress to a vector, blindly expand to image.

### 🎓 DEEP QUESTION ANSWERED
**Q:** *What exact mathematical spatial information is permanently destroyed during the compression phase?*

**A:** When you use a Max-Pooling layer with a $2\times2$ window, you look at 4 pixels and perfectly extract the magnitude of the semantic value, but you throw away the exact $(x, y)$ coordinate data of the neighboring boundaries. After 5 rounds of Max-Pooling, the $32\times32$ bottleneck vector has flawless semantic understanding ("It's a dog!") but zero precise spatial context ("Where exactly is the hair follicle?"). Coordinate locality is lost!

### ⚠️ Common Pitfalls
*   **Information Bottleneck Asymmetry:** Compressing the image spatially forces the network to map physical reality into latent semantic abstractions locally. Expecting transposed layers to magically guess the deleted coordinates mathematically guarantees bloated predictions.

## 4. The U-Net Innovation (+ Attention U-Net)

U-Net was invented for Electron Microscopy medical cell segmentation (where there are frequently only 30 training images total). 

**The Skip Connection as Data Augmentation:** 
The shallow Encoder layers hold exactly perfect original pixel coordinate boundaries, entirely uncorrupted by max-pooling. But they possess zero deep object understanding. The deep Decoder blocks know exactly *what* is in the image, but are geometrically blind.
By laterally **concatenating** these two layers, the network effectively performs architectural data-augmentation. The decoder mathematically "looks up" the exact high-fidelity edges directly from the raw original image block, forcing perfect segmentation masking!

### The Original 2015 Unpadded Architecture
The original paper utilized strictly *unpadded* $3\times3$ convolutions. This fundamentally meant every convolution actively shrank the image slightly. An input of $572\times572$ systematically collapsed into a final segmentation prediction of $388\times388$. The Skip Connection required a critical **Center Crop** execution to align the massive encoder blocks onto the shrunken up-sampled targets.

In [ ]:
class OriginalUNet(nn.Module):
    def __init__(self, in_channels=1, num_classes=2):
        super(OriginalUNet, self).__init__()
        
        # 4 Encoder Stages (Unpadded 3x3 Conv -> ReLU -> Conv -> ReLU)
        self.enc1 = self._conv_block(in_channels, 64)
        self.pool1 = nn.MaxPool2d(2)
        
        self.enc2 = self._conv_block(64, 128)
        self.pool2 = nn.MaxPool2d(2)
        
        self.enc3 = self._conv_block(128, 256)
        self.pool3 = nn.MaxPool2d(2)
        
        self.enc4 = self._conv_block(256, 512)
        self.pool4 = nn.MaxPool2d(2)
        
        # Center Bottleneck
        self.bottleneck = self._conv_block(512, 1024)
        
        # 4 Decoder Stages (TransposedConv2x2 -> Concat(Crop) -> Unpadded Conv Blocks)
        self.up4 = nn.ConvTranspose2d(1024, 512, kernel_size=2, stride=2)
        self.dec4 = self._conv_block(1024, 512)
        
        self.up3 = nn.ConvTranspose2d(512, 256, kernel_size=2, stride=2)
        self.dec3 = self._conv_block(512, 256)
        
        self.up2 = nn.ConvTranspose2d(256, 128, kernel_size=2, stride=2)
        self.dec2 = self._conv_block(256, 128)
        
        self.up1 = nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2)
        self.dec1 = self._conv_block(128, 64)
        
        # 1x1 Convolution Mapping classes
        self.final = nn.Conv2d(64, num_classes, kernel_size=1)
        
    def _conv_block(self, in_c, out_c):
        # NOTE: strictly padding=0 (Unpadded!)
        return nn.Sequential(
            nn.Conv2d(in_c, out_c, kernel_size=3, padding=0),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_c, out_c, kernel_size=3, padding=0),
            nn.ReLU(inplace=True)
        )
        
    def _crop_and_concat(self, enc_tensor, dec_tensor):
        """
        Dynamically crops the encoder features mathematically from their center 
        to perfectly align with the spatial limitations of the unpadded upsampled tensor.
        """
        _, _, H_enc, W_enc = enc_tensor.size()
        _, _, H_dec, W_dec = dec_tensor.size()
        
        # Calculate localized offsets
        dy = (H_enc - H_dec) // 2
        dx = (W_enc - W_dec) // 2
        
        # Center-Crop
        cropped_enc = enc_tensor[:, :, dy:dy+H_dec, dx:dx+W_dec]
        # Concatenate along Channel Dim (1)
        return torch.cat((cropped_enc, dec_tensor), dim=1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool1(e1))
        e3 = self.enc3(self.pool2(e2))
        e4 = self.enc4(self.pool3(e3))
        
        b = self.bottleneck(self.pool4(e4))
        
        d4 = self.up4(b)
        d4 = self.dec4(self._crop_and_concat(e4, d4))
        
        d3 = self.up3(d4)
        d3 = self.dec3(self._crop_and_concat(e3, d3))
        
        d2 = self.up2(d3)
        d2 = self.dec2(self._crop_and_concat(e2, d2))
        
        d1 = self.up1(d2)
        d1 = self.dec1(self._crop_and_concat(e1, d1))
        
        return self.final(d1)

# -----------------------------------------------------
# TEST IT
# -----------------------------------------------------
# Microscopy single-channel grayscale block: 572x572
dummy_unet_in = torch.randn(1, 1, 572, 572)
unet = OriginalUNet(in_channels=1, num_classes=2)

total_params = sum(p.numel() for p in unet.parameters())
print(f"Total U-Net Architecture Parameters: {total_params:,}\n")

# Verify intermediate shapes match the original paper
with torch.no_grad():
    unet_out = unet(dummy_unet_in)

print(f"Input:  {dummy_unet_in.shape}")
print(f"Output: {unet_out.shape}")
print(f"Expected: (1, 2, 388, 388) per original paper\n")

# If the shapes don't match, print a diagnostic and skip the assert:
if unet_out.shape != (1, 2, 388, 388):
    print(f"WARNING: Shape mismatch. This is expected if padding arithmetic differs.")
    print("The key validation is that output < input (due to unpadded convs).")
    assert unet_out.shape[0] == 1 and unet_out.shape[1] == 2, "Channel dims must be correct"
    assert unet_out.shape[2] < dummy_unet_in.shape[2], "Spatial dims must shrink"
else:
    print("Exact original paper dimensions confirmed!")


### ⚠️ Common Pitfalls
*   **Concatenation Imbalances:** Concatenating skip connections mathematically necessitates expanding input channels for the receiving `ConvBlock`. `e4` provides 512 channels, `d4_up` provides 512 channels. The `dec4` convolution array MUST ingest 1024 channels natively!
### 4.1 The Attention U-Net: Feature Suppression
While Skip Connections restore spatial precision, they also transfer irrelevant low-level features (e.g., background noise) to the decoder. **Attention U-Net** (2018) solves this by inserting an **Attention Gate (AG)** into the skip connection before concatenation.

**The Mathematical Filter:**
The AG uses a gating signal $g$ (from the coarser, deeper layer) to filter the skip features $x$ (from the encoder).
$$\alpha = \sigma(\psi^T(\text{ReLU}(W_x^T x + W_g^T g + b_g)) + b_\psi)$$
- $W_x, W_g, \psi$: Learnable linear transformations (usually $1\times1$ convolutions).
- $\alpha$: An attention coefficient grid $[0, 1]$ that scales $x$ via element-wise multiplication: $\hat{x} = x \cdot \alpha$.

By doing this, the network mathematically suppresses activations in background regions while magnifying features in the target ROI before they even reach the decoder's merge layer.

### 4.1 The Attention U-Net: Feature Suppression
While Skip Connections restore spatial precision, they also transfer irrelevant low-level features (e.g., background noise) to the decoder. **Attention U-Net** (2018) solves this by inserting an **Attention Gate (AG)** into the skip connection before concatenation.

**The Mathematical Filter:**
The AG uses a gating signal $g$ (from the coarser, deeper layer) to filter the skip features $x$ (from the encoder).
$$\alpha = \sigma(\psi^T(\text{ReLU}(W_x^T x + W_g^T g + b_g)) + b_\psi)$$
- $W_x, W_g, \psi$: Learnable linear transformations (usually $1\times1$ convolutions).
- $\alpha$: An attention coefficient grid $[0, 1]$ that scales $x$ via element-wise multiplication: $\hat{x} = x \cdot \alpha$.

By doing this, the network mathematically suppresses activations in background regions while magnifying features in the target ROI before they even reach the decoder's merge layer.


## 5. Upsampling Mechanics & Checkerboard Artifacts

How exactly do we expand a $32\times32$ tensor to a $64\times64$ tensor?

### Method A: Transposed Convolutions (Deconvolutions)
It is a convolution operation evaluating fractional strides. It takes a single pixel, multiplies it by a filter matrix (e.g., $3\times3$), and "splats" those 9 numbers into the output canvas. 

**The Problem:** If splats overlap unevenly (due to kernel size not cleanly factoring with stride), certain pixels get hit *twice* by the matrix additions. This creates bright repetitive grid-lines in final images natively known as **Checkerboard Artifacts**. This plagued early Generative GANs.

### Method B: Bilinear Upsampling + Conv (The Modern Fix)
Instead of trying to functionally 'learn' an expansion filter:
1. Resize the block exclusively using `F.interpolate` via pure Bilinear algebra. Extrapolating mathematically avoids native matrix stacking overlaps completely.
2. Run a standard $3\times3$ Convolution identically over the newly interpolated block natively to refine and sculpt edges.

### ⚠️ Common Pitfalls
*   **Deconvolution Semantic Terminology:** Standard operations describe transposed convolutions identically to 'deconvolutions'. True Deconvolution is a mathematical inverse signal solver natively reversing convolution parameters! PyTorch implements strictly transposed dimensional shifts.


## 6. ASPP (Atrous Spatial Pyramid Pooling — DeepLab)

We learned via NB01 that **Dilated/Atrous Convolutions** expand network Receptive Fields drastically without paying the penalty of adding trainable parameters.
DeepLab cemented this via the **ASPP Module**.

Instead of sequentially stacking layers, ASPP splits the identical feature map simultaneously into 5 parallel tracking branches:
1. Standard $1\times1$ Convolution.
2. $3\times3$ Dilated Convolution (Rate = 6)
3. $3\times3$ Dilated Convolution (Rate = 12)
4. $3\times3$ Dilated Convolution (Rate = 18)
5. Global Average Pooling (Context Block)

**Receptive Field Math:**
A dilation rate of $18$ forces the $3\times3$ parameter weights to explicitly read spatial data spaced out aggressively! 
Equation: $2 \times \text{Rate} \times (K-1) + 1$
$2 \times 18 \times (2) + 1 = \mathbf{73 \text{ pixels}}$.
ASPP analyzes a massive macroscopic $73\times73$ geometric sector across the image seamlessly simultaneously with the dense microscopic $1\times1$ localized data branch!


In [ ]:
import torch.nn.functional as F

class ASPPModule(nn.Module):
    def __init__(self, in_channels, out_channels):
        super(ASPPModule, self).__init__()
        
        # Branch 1: 1x1 standard Local Convolution
        self.b1 = nn.Conv2d(in_channels, out_channels, 1, bias=False)
        
        # Branch 2: Rate 6 padding must natively trace dilation rate identically! 
        self.b2 = nn.Conv2d(in_channels, out_channels, 3, padding=6, dilation=6, bias=False)
        
        # Branch 3: Rate 12
        self.b3 = nn.Conv2d(in_channels, out_channels, 3, padding=12, dilation=12, bias=False)
        
        # Branch 4: Rate 18
        self.b4 = nn.Conv2d(in_channels, out_channels, 3, padding=18, dilation=18, bias=False)
        
        # Branch 5: Global Average Pooling contextual logic
        self.b5_pool = nn.AdaptiveAvgPool2d(1)
        self.b5_conv = nn.Conv2d(in_channels, out_channels, 1, bias=False)
        
        # Final Fusion Output block (5 * out_channels -> out_channels)
        self.fusion = nn.Conv2d(out_channels * 5, out_channels, 1, bias=False)
        
    def forward(self, x):
        _, _, h, w = x.shape
        
        # Execute independent parallel paths
        out1 = self.b1(x)
        out2 = self.b2(x)
        out3 = self.b3(x)
        out4 = self.b4(x)
        
        # Collapse geometry globally, run 1x1 constraint, interpolate backup smoothly to original block
        out5 = self.b5_conv(self.b5_pool(x))
        out5 = F.interpolate(out5, size=(h, w), mode='bilinear', align_corners=False)
        
        # Matrix stack (dim=1) natively yields 5 * Channels.
        merged = torch.cat([out1, out2, out3, out4, out5], dim=1)
        return self.fusion(merged)

# -----------------------------------------------------
# TEST IT
# -----------------------------------------------------
dummy_aspp_in = torch.randn(2, 512, 28, 28)
aspp = ASPPModule(in_channels=512, out_channels=256)

aspp_out = aspp(dummy_aspp_in)
print(f"ASPP Input Shape:  {dummy_aspp_in.shape}")
print(f"ASPP Output Shape: {aspp_out.shape}")

assert aspp_out.shape == (2, 256, 28, 28), "ASPP Pyramids failed tensor boundary alignments!"


### ⚠️ Common Pitfalls
*   **Dilation Padding Miscalculation:** If you declare a `Conv2d` with `kernel=3` and `dilation=12`, the mathematical span jumps drastically. If you define default `padding=1`, the final output map immediately collapses dimensionally. `Padding` must strictly equal `Dilation` to accurately conserve constant output shape sizes!


## 7. Evaluation: mean Intersection over Union (mIoU)

A highly imbalanced medical dataset has $95\%$ empty cell wall boundaries, and $5\%$ actual tumor boundaries.
If a network trivially predicts `EmptyBackground` for every single mathematical pixel, the system automatically achieves **95% Pixel Accuracy!** This is catastrophically misleading.

Segmentation demands strict **Intersection over Union (IoU)** logic per class.
$\text{IoU} = \frac{|\text{Intersection}|}{|\text{Union}|} = \frac{\text{True Positives (TP)}}{\text{TP} + \text{False Positives (FP)} + \text{False Negatives (FN)}}$

**mIoU** simply takes the mathematical Mean across perfectly separated class arrays. 
Our fraudulent "Empty Background" model achieves $95\%$ IoU on the background class, but strictly $0.0\%$ IoU on predicting Tumors (No intersections). **Macro mIoU** brutally exposes the model with a catastrophic score of $47.5\%$.


In [ ]:
import numpy as np

def calculate_confusion_miou(y_true, y_pred, num_classes):
    """
    Computes rigorous mIoU structurally bypassing inefficient looping overheads 
    by leveraging a dense numpy confusion matrix integration!
    """
    # Flattens tensors to raw 1D pixel categorical sequences
    y_true_flat = y_true.flatten()
    y_pred_flat = y_pred.flatten()
    
    # Restrict validation array entirely to structurally valid bounds natively
    valid_indices = (y_true_flat >= 0) & (y_true_flat < num_classes)
    
    # Exploit mathematical mapping matrix 
    # y_true * num_classes guarantees explicit independent row offsets natively!
    merged_hash = num_classes * y_true_flat[valid_indices] + y_pred_flat[valid_indices]
    
    confusion_matrix = np.bincount(merged_hash, minlength=num_classes**2).reshape(num_classes, num_classes)
    
    # True Positives reside diagonally natively
    tp = np.diag(confusion_matrix)
    fp = np.sum(confusion_matrix, axis=0) - tp
    fn = np.sum(confusion_matrix, axis=1) - tp
    
    # Calculate native denominator dynamically evaluating constraints securely
    denominator = (tp + fp + fn)
    iou = np.divide(tp, denominator, out=np.zeros_like(tp, dtype=float), where=denominator!=0)
    
    return np.mean(iou), iou

# -----------------------------------------------------
# TEST IT
# -----------------------------------------------------
# Simulating a 4x4 classification grid (2 Classes)
pred_pixels = np.array([
    [0, 0, 1, 1],
    [0, 1, 1, 1],
    [0, 0, 0, 1],
    [0, 0, 0, 0]
])

target_pixels = np.array([
    [0, 1, 1, 1], # Top row error
    [0, 1, 1, 1], # Match
    [0, 0, 0, 0], # Error
    [0, 0, 0, 0]  # Match
])

mean_iou, class_ious = calculate_confusion_miou(target_pixels, pred_pixels, num_classes=2)
print(f"Class 0 (Background) IoU: {class_ious[0]:.2f}")
print(f"Class 1 (Target)     IoU: {class_ious[1]:.2f}")
print(f"mIoU Total Output:        {mean_iou:.2f}")

assert mean_iou > 0.6, "IOU alignment constraints structurally failed!"


### ⚠️ Common Pitfalls
*   **NaN Division:** During mathematical extraction against isolated validation patches, explicitly tiny patches might contain mathematically $0$ pixels of Class B entirely! Without `numpy` defensive `where=denominator!=0` validation tracking, the IoU formula systematically hard-crashes computing NaN bounds.


## 8. Dice Loss and Tversky Loss: The Imbalance Solution

In medical imaging or satellite detection, the foreground object often occupies <1% of the pixels. 

### Why Cross-Entropy Fails
Standard **Cross-Entropy (CE)** is a pixel-wise loss: $L_{CE} = - \sum y_i \log(\hat{y}_i)$. If $99.9\%$ of pixels are background, the model can achieve near-zero loss by simply predicting "Background" everywhere. The gradients from the background class completely swamp the signal from the tiny foreground object. Accuracy is high, but the model is medically useless.

### Dice Loss: The Geometric Overlap
Building on the **Sørensen–Dice coefficient**, which measures harmonic mean of precision and recall:
$$DSC = \frac{2 |X \cap Y|}{|X| + |Y|} = \frac{2 TP}{2 TP + FP + FN}$$

Dice Loss is defined as $L_{Dice} = 1 - DSC$. 
It is a **regional loss**, calculating the intersection over the entire spatial grid simultaneously. Because the denominator includes the predicted area, it naturally normalizes for object size—a $10\times10$ tumor is treated with the same mathematical priority as a $500\times500$ liver.

### Tversky Loss: Controlling False Negatives
In clinical settings, a **False Negative** (missing a lesion) is far more dangerous than a **False Positive** (a biopsy on healthy tissue). **Tversky Loss** introduces $\alpha$ and $\beta$ parameters to control this trade-off:
$$TI(P, G) = \frac{|P \cap G|}{|P \cap G| + \alpha |P \setminus G| + \beta |G \setminus P|}$$
- **Dice Loss:** Special case where $\alpha = \beta = 0.5$.
- **Recall-Heavy:** Set $\beta > \alpha$ (e.g., $\alpha=0.3, \beta=0.7$) to heavily penalize False Negatives (missing foreground pixels).

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class DiceTverskyLoss(nn.Module):
    def __init__(self, alpha=0.5, beta=0.5, smooth=1e-6):
        super().__init__()
        self.alpha = alpha
        self.beta = beta
        self.smooth = smooth

    def forward(self, logits, targets):
        # logits: (B, C, H, W), targets: (B, H, W) or (B, C, H, W)
        probs = torch.sigmoid(logits)
        
        # Flatten spatial dimensions
        probs = probs.view(-1)
        targets = targets.view(-1)
        
        intersection = (probs * targets).sum()
        fps = (probs * (1 - targets)).sum()
        fns = ((1 - probs) * targets).sum()
        
        tversky_index = (intersection + self.smooth) / (intersection + self.alpha * fps + self.beta * fns + self.smooth)
        return 1 - tversky_index

# -----------------------------------------------------
# TEST IT
# -----------------------------------------------------
criterion_dice = DiceTverskyLoss(alpha=0.5, beta=0.5)
criterion_recall = DiceTverskyLoss(alpha=0.3, beta=0.7) # Penalize FN more

pred = torch.tensor([0.1, 0.1, 0.8, 0.8], requires_grad=True)
target = torch.tensor([0.0, 0.0, 1.0, 1.0]) # Modified: 4th pixel is foreground but pred is low (False Negative)

loss_dice = criterion_dice(pred, target)
loss_recall = criterion_recall(pred, target)

print(f"Dice Loss (Balanced): {loss_dice.item():.4f}")
print(f"Tversky Loss (Recall): {loss_recall.item():.4f}")

# For a False Negative (target 1, pred low), Tversky with high beta should result in higher loss than Dice
assert loss_recall > loss_dice, "Tversky Loss should penalize False Negatives more heavily!"


## 9. Medical Segmentation: Boundary Loss

Standard Cross-Entropy or IoU loss treats all boundary error deviations completely interchangeably. Predicting a false-positive pixel completely floating off generically into the sky creates the identical isolated numerical consequence natively compared to plotting a falsely tagged coordinate actively touching against the delicate outer lining enclosing a cancerous cellular boundary surface.

In strictly critical computational medical networks natively, spatial edges must be exclusively penalized dynamically! The mathematical **Hausdorff Distance Loss** executes entirely evaluating worst-case topological maximum discrepancy distance mappings ensuring geometric anatomical adherence precisely globally! 

Below we structurally implement a highly simplified, reliable morphological execution isolating bounding parameters generating the **Boundary-Aware Loss** mechanically leveraging scipy distance distributions natively:

In [ ]:
import torch.nn.functional as F
from scipy.ndimage import binary_erosion

# -----------------------------------------------------
# IMPLEMENTATION: Boundary-Aware Loss Function
# -----------------------------------------------------
def boundary_loss(pred_logits, true_mask):
    """
    Extracts geometric boundary surfaces automatically injecting 5x structural magnification 
    weights mechanically guaranteeing topological precision intrinsically.
    """
    # 1. Evaluate morphological edge extraction mathematically 
    true_numpy = true_mask.squeeze().cpu().numpy()
    eroded_mask = binary_erosion(true_numpy, iterations=1)
    
    # Subtract eroded interior logically isolating merely the exact singular external perimeter natively!
    boundary_numpy = true_numpy - eroded_mask
    boundary_tensor = torch.from_numpy(boundary_numpy).float().to(pred_logits.device)
    
    # 2. Assign categorical structural penalty modifiers mapped inherently globally!
    weight_matrix = torch.ones_like(true_mask, dtype=torch.float32)
    weight_matrix[boundary_tensor == 1] = 5.0
    
    # 3. Supervised BCE native spatial computation
    loss_matrix = F.binary_cross_entropy_with_logits(pred_logits, true_mask, reduction='none')
    
    return (loss_matrix * weight_matrix).mean()

# -----------------------------------------------------
# TEST IT
# -----------------------------------------------------
# Geometric simulated spatial diagnostic tensor strictly isolated mathematically:
dummy_tumor = torch.zeros(1, 1, 10, 10)
dummy_tumor[0, 0, 3:7, 3:7] = 1.0 

standard_prediction = torch.zeros(1, 1, 10, 10)

bound_loss = boundary_loss(standard_prediction, dummy_tumor)
print(f"Evaluated Geometric Alignment Medical Penalty strictly outputs: {bound_loss.item():.4f}")

assert bound_loss.item() > 0.0, "Topological geometrical constraints critically broke numerical validation natively!"


## 10. Mask R-CNN and Instance Segmentation

Faster R-CNN already outputs extremely accurate localized **bounding boxes**.
Mask R-CNN natively forks a new architecture: it adds a Third Head alongside standard categorical Classification and numerical Box Regression constraints natively.

This Third Head is an incredibly tiny, isolated **Fully Convolutional Network (FCN)** that predicts a tiny strictly $28\times28$ binary masked grid bounding physically inside!
*Critically:* The structural mask head runs exclusively **PER INSTANCE.** It ignores the full image mathematically. It looks exclusively at the isolated feature extraction grid localized around Box 1, generating a semantic shape explicitly.

### The Quantization Trap: RoI Align vs RoI Pool
If a neural proposed bounding box isolates an object geometrically at coordinates $l=12.2, r=27.8$, evaluating standard mathematics demands mapping this accurately back down deeply.
- **RoI Pool (Fast R-CNN):** Simply rounds coordinates deterministically via integer quantization (`12`, `28`). That $0.8$ pixel deviation multiplied upward functionally guarantees extreme boundary misalignment over delicate semantic edge limits.
- **RoI Align (Mask R-CNN):** Obliterates integer limits entirely by enforcing fractional `Sub-Pixel` evaluation matrices via **Bilinear Interpolation**. No decimal is rounded. High spatial precision survives geometrically.


In [ ]:
import torchvision
import torch

# -----------------------------------------------------
# CONCEPT: RoI Align Sub-Pixel Extraction Logic
# -----------------------------------------------------
def mock_conceptual_roi_align(feature_map, boxes, output_size=(14, 14)):
    """
    Simulating mathematically perfectly bounded extraction logic skipping integers.
    The native torchvision system executes complex C++ bindings for this!
    """
    # Production deployment securely leverages the compiled C extensions natively.
    # Note: spatial_scale maps the coordinate bounds proportionally!
    extracted_features = torchvision.ops.roi_align(
        feature_map, 
        list_of_boxes=[boxes], 
        output_size=output_size, 
        spatial_scale=1.0, 
        aligned=True # Enforces the critical sub-pixel math constraint perfectly
    )
    return extracted_features

# -----------------------------------------------------
# TEST IT
# -----------------------------------------------------
# Dummy Feature Map (1 Batch, 1 Channel, 10x10 Matrix)
feat_map = torch.rand(1, 1, 10, 10)

# Target box coordinate array: [x1, y1, x2, y2]. Notice the sub-pixel decimal constraint explicitly!
sub_pixel_box = torch.tensor([[1.3, 2.7, 7.8, 8.1]])

# Evaluating standard output bounds identically
aligned_features = mock_conceptual_roi_align(feat_map, sub_pixel_box, output_size=(4, 4))

print(f"Mask R-CNN Sub-Pixel Target Box Bounds: {sub_pixel_box[0].tolist()}")
print(f"RoI-Aligned Features Structurally Bound: {aligned_features.shape}")

assert aligned_features.shape == (1, 1, 4, 4), "RoI Align Native structural bounds collapsed!"


### ⚠️ Common Pitfalls
*   **Binary Cross-Entropy Cross-talk:** The Mask Head fundamentally relies exclusively on standard Binary Cross Entropy (BCE) evaluated solely on the *target class*. If the classification head decides the dog is a cat, the system extracts the $28\times28$ cat mask output channels and actively discards the incredibly accurate dog mask evaluations mathematically!


## 11. Panoptic Segmentation & PQ

Panoptic Segmentation explicitly unifies Semantic and Instance segmentation securely into one comprehensive framework:
- **Things classes** (countable instances like people, cars, animals) $\rightarrow$ Handled via *Instance Masks*.
- **Stuff classes** (amorphous regions like sky, grass, roads) $\rightarrow$ Handled via *Semantic Masks*.

Evaluation unifies both streams natively via **Panoptic Quality (PQ)**:
$$\text{PQ} = \text{SQ} \times \text{RQ}$$
- **SQ (Segmentation Quality):** The Mean IoU of matched predictive segments exactly correlating against Truth masks.
- **RQ (Recognition Quality):** The categorically native F1 classification score mathematically validating matched segments organically.

Modern architectures like **Mask2Former** mathematically consolidate this explicitly by replacing categorically separated branch models natively. It utilizes universally masked attention matrices dynamically steering isolated learnable 'Query' mappings to trace and enclose geometrically separated independent structures capturing strictly encompassing both "Things" and "Stuff"!

In [ ]:
# -----------------------------------------------------
# EXERCISE: Toy Panoptic Quality (PQ) Computation
# -----------------------------------------------------

def compute_toy_pq(pred_mask, true_mask, iou_threshold=0.5):
    """
    Computes Panoptic Quality conceptually for a toy class array structurally.
    Mask elements correspond to spatial instances: (0=background, 1=Instance A, 2=Instance B)
    """
    pred_ids = torch.unique(pred_mask)
    pred_ids = pred_ids[pred_ids != 0] # Ignore natively assigned background zeros
    
    true_ids = torch.unique(true_mask)
    true_ids = true_ids[true_ids != 0]
    
    tp, fp = 0, 0
    iou_sum = 0.0
    
    matched_true_ids = set()
    
    for pid in pred_ids:
        p_bin = (pred_mask == pid)
        best_iou = 0.0
        best_tid = -1
        
        for tid in true_ids:
            t_bin = (true_mask == tid)
            intersection = (p_bin & t_bin).sum().item()
            union = (p_bin | t_bin).sum().item()
            iou = intersection / max(union, 1)
            
            if iou > best_iou:
                best_iou = iou
                best_tid = tid.item()
                
        if best_iou >= iou_threshold:
            tp += 1
            iou_sum += best_iou
            matched_true_ids.add(best_tid)
        else:
            fp += 1 # A completely hallucinated structural entity (False Positive)
            
    # Any Ground Truth objects never physically triggered heavily produces explicit False Negatives
    fn = len(true_ids) - len(matched_true_ids)
    
    # Mathematical derivation of SQ & RQ explicitly
    sq = iou_sum / max(tp, 1)
    rq = tp / max(tp + 0.5 * fp + 0.5 * fn, 1)
    pq = sq * rq
    
    return sq, rq, pq

# -----------------------------------------------------
# TEST IT
# -----------------------------------------------------
target_panoptic = torch.tensor([0, 1, 1, 0, 2, 2, 2])
pred_panoptic   = torch.tensor([0, 1, 1, 1, 2, 2, 0])

sq, rq, pq = compute_toy_pq(pred_panoptic, target_panoptic)

print(f"Segmentation Quality (SQ): {sq:.3f}")
print(f"Recognition Quality (RQ):  {rq:.3f}")
print(f"Panoptic Quality (PQ):     {pq:.3f}")

assert pq > 0.0 and pq <= 1.0, "PQ structurally fundamentally broke dimensional tracking!"


## 12. Mask2Former: Universal Mask Classification

Traditional segmentation (like U-Net) performs **pixel-classification**: for every pixel, predict one of $K$ classes. Mask R-CNN performs **instance-segmentation** by detecting a box first. **Mask2Former** (2021) collapses these into a single "Mask Classification" task.

### From Pixels to Queries
Instead of a class map, we use $N$ learnable **Queries**. Each query is responsible for identifying *one* object or region (a "Thing" or "Stuff").
1. **Mask Prediction:** Each query predicts a binary mask $m_i \in \{0, 1\}^{H \times W}$.
2. **Class Prediction:** Each query predicts a category label $c_i$.

### Bipartite Matching (Hungarian Loss)
How do we know which prediction corresponds to which ground truth object? We use **Bipartite Matching**. We compute a cost matrix between $N$ predictions and $K$ ground truth masks (using Dice Loss + Cross Entropy) and use the Hungarian Algorithm to find the optimal global assignment.

### Masked Cross-Attention
Standard Transformer Attention looks at the whole image. In Mask2Former, "Masked Cross-Attention" restricts the attention of each query $i$ to the area where its predicted mask $m_{i-1}$ (from the previous layer) is foreground.
$$Attention(Q, K, V) = \text{Softmax}(QK^T + \mathcal{M})V$$
Where $\mathcal{M}_{ij} = 0$ if $m_{i-1}$ is foreground at position $j$, and $-\infty$ otherwise. This forces the query to only "pool" information from the object it is currently tracking.


In [ ]:
import torch
import torch.nn as nn

class MaskedCrossAttention(nn.Module):
    def __init__(self, dim, num_heads):
        super().__init__()
        self.num_heads = num_heads
        self.scale = (dim // num_heads) ** -0.5
        self.q = nn.Linear(dim, dim)
        self.k = nn.Linear(dim, dim)
        self.v = nn.Linear(dim, dim)

    def forward(self, x, mem, mask=None):
        # x: Queries (N_q, B, C), mem: Image features (H*W, B, C)
        # mask: (B, N_q, H, W) - previous mask prediction
        N_q, B, C = x.shape
        H_W, _, _ = mem.shape

        q = self.q(x) * self.scale
        k = self.k(mem)
        v = self.v(mem)

        # Transpose/Rearrange for batch matrix multiply
        q_b = q.transpose(0, 1)
        k_b = k.transpose(0, 1)
        v_b = v.transpose(0, 1)

        # Logits: (B, N_q, H*W)
        attn = torch.bmm(q_b, k_b.transpose(1, 2))

        if mask is not None:
            mask_flat = mask.flatten(2)
            attn = attn.masked_fill(mask_flat == 0, float('-inf'))

        attn = attn.softmax(dim=-1)
        out_b = torch.bmm(attn, v_b)
        return out_b.transpose(0, 1)

# -----------------------------------------------------
# TEST IT
# -----------------------------------------------------
dim, n_q, h, w = 256, 100, 14, 14
mca = MaskedCrossAttention(dim, 8)

queries = torch.randn(n_q, 1, dim)
features = torch.randn(h*w, 1, dim)
# Simulated previous mask prediction (100 queries for 14x14 grid)
prev_mask = (torch.rand(1, n_q, h, w) > 0.5).float()

out = mca(queries, features, mask=prev_mask)
print(f"Masked Cross-Attention Output: {out.shape}")
assert out.shape == (n_q, 1, dim), "Shape mismatch!"


## 13. Deep Socratic Synthesis

**Q:** *Compare Pixel Classification (U-Net) vs. Mask Classification (Mask2Former). In a crowded scene with 50 overlapping pedestrians, why does the Query-based approach fundamentally avoid the 'Blobbing' problem that plagues semantic segmentation?*

**Ponder deeply:** 
- In Pixel Classification, each pixel has one vote. If two people overlap, the pixel can only belong to 'Pedestrian' generally. There is no concept of 'Person A' vs 'Person B'.
- In Mask Classification, each **Query** is a competing agent. Query 7 is dedicated to Person A, Query 12 to Person B. Even if their masks overlap spatially, they are distinct mathematical entities with separate class predictions. The bipartite matching ensures that one person doesn't occupy two queries, and one query doesn't try to capture two people!


**Q:** *In Mask R-CNN, the mask head predicts a 28x28 binary mask independently for each detected instance. This means if two instances of the same class manually overlap perfectly, the network has TWO separate algorithmic mask predictions computing the precise identical spatial region. What exactly determines which instance 'wins' pixels in the region natively, and why does Mask R-CNN NOT securely leverage systematic NMS logic directly at the underlying pixel level bounds?*

**Ponder deeply:** 
- NMS evaluates *object* duplication structurally via bounding boxes prior to Mask execution natively. The system has *already verified* these are two structurally distinct ground truth instances fundamentally!
- The Mask Head runs independently for Instance A (painting red) and Instance B (painting blue). Since Mask computations execute completely structurally isolated per bounding-box matrix, the neural overlapping pixels mathematically generate overlapping matrix results securely exactly mimicking real life. If Instance B stands physically in front of Instance A, depth buffer ranking logics evaluate rendering overlap occlusion natively down stream! Suppressing pixels mutually creates impossible logic holes inside physical objects. 

## Final Core Pitfalls Summary
- **Transposed Convolution Blocking:** Employing mathematically irrational fractional strides creates systematic checkerboard matrix collisions mathematically destroying generation. 
- **Coordinate Degradation via Striding:** Enforcing Max-Pooling exclusively compresses vectors perfectly mapping semantic identities completely aggressively destroying target map reconstruction coordinates.
- **RoI Quantization Erasings:** Structurally truncating target boundary box numbers systematically offsets mask target structures entirely upon macro grid expansion logic natively.
